# Szyfrowanie homomorficzne

Szyfrowanie homomorficzne to rodzaj szyfrowania z kluczem publicznym, w którym **deformowalność szyfrogramów** to pożądana cecha kryptosystemu. Przepomnijmy, że deformowalność szyfrogramów to własność polegająca na możliwości modyfikacji szyfrogramów w taki sposób, że po odszyfrowaniu tak zmodyfikowanego szyfrogramu otrzymujemy tekst jawny, który jest **sensownie** powiązany z oryginalnym tekstem jawnym.

Innymi słowy, szyfrowanie homomorficzne umożliwia wykonywanie operacji (matematycznych) na zaszyfrowanych danych, tak by pod odszyfrowaniu otrzymać wynik tak samo poprawny jak w sytuacji gdy operacją wykonywane na jawnych danych. 

Rozróżniamy kilka poziomów szyfrowania homomorficznego:
1. Szyfrowanie częściowo homomorficzne - kryptosystem wspiera tylko jedną operację
2. _Somewhat Homomorphic Encryption_ - kryptosystem wspiera obie operacje, ale sekwencja możliwych operacji jest ograniczona jeśli chodzi o długość.
3. Szyfrowanie w pełni homomorficzne (FHE - _Fully Homomorphic Encryption_) - pełna obsługa bez ograniczeń operacji na zaszyfrowanych danych. 

Szyfrowanie homomorficzne jest szczególnie pożądane w sytuacji gdy powierzamy swoje dane do przetwarzania obcemu podmiotowi.

In [1]:
## Funkcje pomocnicze
def gcd(a, b):
    while a != 0:
        a, b = b % a, a
    return b

def lcm(a, b):
    return a * b // gcd(a, b)

def findModInverse(a, m):
    if gcd(a, m) != 1:
        return None

    u1, u2, u3 = 1, 0, a
    v1, v2, v3 = 0, 1, m
    while v3 != 0:
        q = u3 // v3
        v1, v2, v3, u1, u2, u3 = (u1 - q * v1), (u2 - q * v2), (u3 - q * v3), v1, v2, v3
    return u1 % m

### Zadanie 1
Dziełając wyłącznie na szyfrogramach RSA zaproponuj i zaimplementuje metodą fałszującą odszyfrowaną wiadomość tak by była ona liczą podzielną przez 17.  

In [2]:
def generateRSAKeys(keySize):
    print('1. Generujemy liczby p i q')
    p=49109
    q=40639
    e = 65537
    print('3. Obliczamy wykładnik prywatny: odwrotność e modulo (p-1)(q-1c)')
    d=findModInverse(e, (p-1)*(q-1))
    n= p*q
    publicKey = (n, e)
    privateKey = (n, d)
    print('Klucz publiczny:', publicKey)
    print('Klucz prywatny:', privateKey)
    return (publicKey, privateKey)

public, private = generateRSAKeys(16)

print('Generujemy klucze publiczny i prywatny:', public, private)

def encryptRSA(data_number, modulus, exp):
    data_encrypted = pow(data_number, exp, modulus)
    return data_encrypted

def decryptRSA(data_number, modulus, exp):
    data_decrypted = pow(data_number, exp, modulus)
    return data_decrypted

def forgeRSACiphertext(ciphertext, modulus, public_exp, target_divisor=17):
    forged = (ciphertext * pow(target_divisor, public_exp, modulus)) % modulus
    return forged

print("\n=== Zadanie 1: Fałszowanie szyfrogramu RSA ===")
message = 12345
n, e = public
n_priv, d = private

cipher = encryptRSA(message, n, e)
print(f"Oryginalna wiadomość: {message}")
print(f"Szyfrogram: {cipher}")

decrypted_original = decryptRSA(cipher, n, d)
print(f"Odszyfrowana oryginalna wiadomość: {decrypted_original}")

forged_cipher = forgeRSACiphertext(cipher, n, e, 17)
print(f"Fałszowany szyfrogram: {forged_cipher}")

decrypted_forged = decryptRSA(forged_cipher, n, d)
print(f"Odszyfrowana fałszowana wiadomość: {decrypted_forged}")
print(f"Czy jest podzielna przez 17? {decrypted_forged % 17 == 0}")
print(f"Wynik: {decrypted_forged} = {decrypted_forged // 17} * 17")


1. Generujemy liczby p i q
3. Obliczamy wykładnik prywatny: odwrotność e modulo (p-1)(q-1c)
Klucz publiczny: (1995740651, 65537)
Klucz prywatny: (1995740651, 601158737)
Generujemy klucze publiczny i prywatny: (1995740651, 65537) (1995740651, 601158737)

=== Zadanie 1: Fałszowanie szyfrogramu RSA ===
Oryginalna wiadomość: 12345
Szyfrogram: 1570446469
Odszyfrowana oryginalna wiadomość: 12345
Fałszowany szyfrogram: 1356740241
Odszyfrowana fałszowana wiadomość: 209865
Czy jest podzielna przez 17? True
Wynik: 209865 = 12345 * 17


## Kryptosystem Pailliera

W 1999 roku Pascal Paillier zaproponował kryptosystem, którego bezpieczeństwo oparte było o problem faktoryzacji liczb całkowitych i problem logarytmu dyskretnego.


#### Generowanie kluczy (wersja podstawowa)

1. Wybierz losowo dwie duże liczby pierwsze $p$ i $q$ tak aby $gcd(pq, (p-1)(q-1)) = 1$.
2. Oblicz $n = pq$
3. Oblicz $\lambda = lcm(p-1,q-1)$ ($lcm$ --- Least Common Multiple, Najmniejsza Wspólna Wielokrotność)
4. Zdefiniuj funkcje $L(x) = \frac{x-1}{n}$
5. Wybierz losowo $g \in Z^*_{n^2}$ (liczba całkowita w zakresie 1 do $n^2$)
6. Oblicz odwrotność multiplikatywną $\mu = L(g^\lambda \bmod n^2))^{-1} \bmod n$. Jeśli $\mu$ nie istnieje zacznij od nowa.

Klucz publiczny ma postać: $pk = (n,g)$
Klucz prywatny ma postać: $sk = (\lambda, \mu)$

#### Generowanie kluczy wersja uproszczona
Jeśli $p$ i $q$ są podobniej długości można użyć prostszego wariantu:
1. $g = n+1$
2. $\lambda = \phi(n)$
3. $\mu = \phi(n)^{-1} \bmod n$

#### Szyfrowanie

1. Tekstem jawnym jest liczba $m$ zakresu $0 \le m < n$.

2. Wybierz losową liczbę z zakresu $0 \le r < n$ oraz względnie pierwszą z $n$

3. Oblicz szyfrogram $c= g^m \cdot r^n \bmod n^2$

#### Deszyfrowanie

1. Szyfrogram musi być liczbą z zakresu $0 < c < n^2$
2. Oblicz tekst jawny $m= L(c^\lambda \bmod n^2)\cdot \mu \bmod n$

### Homomorficzne własności schematu Pailliera

1. Dodawanie dwóch liczb:
$$D_{priv}(E_{pub}(m_1) \cdot E_{pub}(m_2) \bmod n^2)= m_1 + m_2 \bmod n$$

2. Mnożenie szyfrogramu przez liczbę:
$$D_{priv}(E_{pub}(m_1)^{m_2} \bmod n^2)= m_1 \cdot m_2 \bmod n$$

### Zadanie 2
Zaimplementuj szyfrowanie i deszyfrowanie Pailliera oraz funkcje umożliwiające homomorficzne operacja dodawania oraz mnożenia przez liczbę całkowitą. Wykaż poprawność operacji homomorficznych.

In [3]:
import random

def genPaillierKeys():
    p = 49109
    q = 40639
    nt = p * q
    gt = nt + 1
    
    phi_n = (p - 1) * (q - 1)
    lambdat = phi_n
    
    mut = findModInverse(phi_n, nt)
    
    return nt, gt, lambdat, mut

n, g, Lambda, Mu = genPaillierKeys()
print(f"Klucz publiczny Pailliera: n={n}, g={g}")
print(f"Klucz prywatny Pailliera: Lambda={Lambda}, Mu={Mu}")

def L(x, n):
    return (x - 1) // n

def encryptPaillier(m, n, g):
    while True:
        r = random.randint(1, n - 1)
        if gcd(r, n) == 1:
            break
    
    g_m = pow(g, m, n * n)
    r_n = pow(r, n, n * n)
    c = (g_m * r_n) % (n * n)
    return c

def decryptPaillier(c, Lambda, Mu, n):
    c_lambda = pow(c, Lambda, n * n)
    l_value = L(c_lambda, n)
    m = (l_value * Mu) % n
    return m

print("\n=== Test podstawowego szyfrowania/deszyfrowania Pailliera ===")
test_message = 12345
encrypted = encryptPaillier(test_message, n, g)
decrypted = decryptPaillier(encrypted, Lambda, Mu, n)
print(f"Wiadomość: {test_message}")
print(f"Zaszyfrowana: {encrypted}")
print(f"Odszyfrowana: {decrypted}")
print(f"Poprawność: {test_message == decrypted}")

def homomorphic_add(c1, c2, n):
    return (c1 * c2) % (n * n)

def homomorphic_multiply(c, k, n):
    return pow(c, k, n * n)

print("\n=== Test operacji homomorficznych ===")
m1 = 50
m2 = 30
k = 3

c1 = encryptPaillier(m1, n, g)
c2 = encryptPaillier(m2, n, g)

print(f"m1 = {m1}, m2 = {m2}, k = {k}")

c_sum = homomorphic_add(c1, c2, n)
m_sum = decryptPaillier(c_sum, Lambda, Mu, n)
print(f"\nDodawanie homomorficzne:")
print(f"  E(m1) * E(m2) = E({m1} + {m2} mod n)")
print(f"  Odszyfrowany wynik: {m_sum}")
print(f"  Oczekiwany wynik: {(m1 + m2) % n}")
print(f"  Poprawność: {m_sum == (m1 + m2) % n}")

c_mult = homomorphic_multiply(c1, k, n)
m_mult = decryptPaillier(c_mult, Lambda, Mu, n)
print(f"\nMnożenie homomorficzne:")
print(f"  E(m1)^k = E({m1} * {k} mod n)")
print(f"  Odszyfrowany wynik: {m_mult}")
print(f"  Oczekiwany wynik: {(m1 * k) % n}")
print(f"  Poprawność: {m_mult == (m1 * k) % n}")


Klucz publiczny Pailliera: n=1995740651, g=1995740652
Klucz prywatny Pailliera: Lambda=1995650904, Mu=123684463

=== Test podstawowego szyfrowania/deszyfrowania Pailliera ===
Wiadomość: 12345
Zaszyfrowana: 934145669619141521
Odszyfrowana: 12345
Poprawność: True

=== Test operacji homomorficznych ===
m1 = 50, m2 = 30, k = 3

Dodawanie homomorficzne:
  E(m1) * E(m2) = E(50 + 30 mod n)
  Odszyfrowany wynik: 80
  Oczekiwany wynik: 80
  Poprawność: True

Mnożenie homomorficzne:
  E(m1)^k = E(50 * 3 mod n)
  Odszyfrowany wynik: 150
  Oczekiwany wynik: 150
  Poprawność: True


## Szyfrowanie Homomorficzne

Rozważmy następujący naiwny kryptosystem :

#### Generowanie kluczy 
1. wybierz dużą liczbę nieparzystą $p$
2. wybierz ogranicznie szumu, znacznie mniejsze od $p$: $B << p$

#### Szyfrowanie 
1. szyfrujemy wiadomości 1-bitowe: $m$
2. wylosuj duże $q$ (porównywalne z $p$)
3. wylosuj $r \in \{0,...,B\}$
3. oblicz $c=m+2r + pq$

#### Deszyfrowanie
1. oblicz $t=c \bmod p$
2. oblicz $md = t \bmod 2$

#### Operacje homomorficzne):
1. Dodawania:$c_+ = c_1 + c_2 $
2. Mnożenie: $c_* = c_1 \cdot c_2$


### Zadanie 3
Zaimplementuj opisany powyżej trywialny kryptosystem. Sprawdź poprawność działania dla obu operacji. Możesz zastosować następujące parametry:
- $p = 29$
- $q \approx 22$
- $r = 3$

Sprawdź jak będzie dział ten kryptosystem gdy dodasz do siebie trzy liczby jedna po drugiej np.: $c_1(0) + c_2(0) +c_3(0)$

In [4]:
def generate_trivial_keys(p, B):
    return p, B

def encrypt_trivial(m, p, q, r, B):
    if m not in [0, 1]:
        raise ValueError("Wiadomość musi być 1-bitowa (0 lub 1)")
    if r > B:
        raise ValueError(f"r musi być <= B ({B})")
    
    c = m + 2 * r + p * q
    return c

def decrypt_trivial(c, p):
    t = c % p
    md = t % 2
    return md

def homomorphic_add_trivial(c1, c2):
    return c1 + c2

def homomorphic_multiply_trivial(c1, c2):
    return c1 * c2

print("=== Zadanie 3: Trywialny kryptosystem homomorficzny ===")
p = 29
q = 22
r = 3
B = 10

print(f"Parametry: p={p}, q={q}, r={r}, B={B}")

print("\n--- Test podstawowego szyfrowania/deszyfrowania ---")
for m in [0, 1]:
    c = encrypt_trivial(m, p, q, r, B)
    md = decrypt_trivial(c, p)
    print(f"m={m} -> c={c} -> md={md} (poprawność: {m == md})")

print("\n--- Test operacji homomorficznych ---")

m1, m2 = 0, 1
c1 = encrypt_trivial(m1, p, q, r, B)
c2 = encrypt_trivial(m2, p, q, r, B)
c_sum = homomorphic_add_trivial(c1, c2)
m_sum = decrypt_trivial(c_sum, p)
print(f"Dodawanie: E({m1}) + E({m2}) = E({m1} + {m2} mod 2)")
print(f"  c1={c1}, c2={c2}, c_sum={c_sum}")
print(f"  Odszyfrowany wynik: {m_sum}, oczekiwany: {(m1 + m2) % 2}")
print(f"  Poprawność: {m_sum == (m1 + m2) % 2}")

c_mult = homomorphic_multiply_trivial(c1, c2)
m_mult = decrypt_trivial(c_mult, p)
print(f"\nMnożenie: E({m1}) * E({m2}) = E({m1} * {m2} mod 2)")
print(f"  c1={c1}, c2={c2}, c_mult={c_mult}")
print(f"  Odszyfrowany wynik: {m_mult}, oczekiwany: {(m1 * m2) % 2}")
print(f"  Poprawność: {m_mult == (m1 * m2) % 2}")

print("\n--- Test dodawania trzech liczb: c1(0) + c2(0) + c3(0) ---")
m1, m2, m3 = 0, 0, 0
c1 = encrypt_trivial(m1, p, q, r, B)
c2 = encrypt_trivial(m2, p, q, r, B)
c3 = encrypt_trivial(m3, p, q, r, B)

c_total = homomorphic_add_trivial(homomorphic_add_trivial(c1, c2), c3)
m_total = decrypt_trivial(c_total, p)
print(f"E(0) + E(0) + E(0) = E({m1} + {m2} + {m3} mod 2)")
print(f"  c1={c1}, c2={c2}, c3={c3}")
print(f"  c_total={c_total}")
print(f"  Odszyfrowany wynik: {m_total}, oczekiwany: {(m1 + m2 + m3) % 2}")
print(f"  Poprawność: {m_total == (m1 + m2 + m3) % 2}")

print("\n--- Analiza problemu z wielokrotnym dodawaniem ---")
print("Problem: Po wielokrotnym dodawaniu wartość c może przekroczyć zakres,")
print("co może prowadzić do błędów w deszyfrowaniu.")
print(f"Przykład: c1={c1}, po trzech dodaniach: c_total={c_total}")
print(f"c_total mod p = {c_total % p}, (c_total mod p) mod 2 = {m_total}")

=== Zadanie 3: Trywialny kryptosystem homomorficzny ===
Parametry: p=29, q=22, r=3, B=10

--- Test podstawowego szyfrowania/deszyfrowania ---
m=0 -> c=644 -> md=0 (poprawność: True)
m=1 -> c=645 -> md=1 (poprawność: True)

--- Test operacji homomorficznych ---
Dodawanie: E(0) + E(1) = E(0 + 1 mod 2)
  c1=644, c2=645, c_sum=1289
  Odszyfrowany wynik: 1, oczekiwany: 1
  Poprawność: True

Mnożenie: E(0) * E(1) = E(0 * 1 mod 2)
  c1=644, c2=645, c_mult=415380
  Odszyfrowany wynik: 1, oczekiwany: 0
  Poprawność: False

--- Test dodawania trzech liczb: c1(0) + c2(0) + c3(0) ---
E(0) + E(0) + E(0) = E(0 + 0 + 0 mod 2)
  c1=644, c2=644, c3=644
  c_total=1932
  Odszyfrowany wynik: 0, oczekiwany: 0
  Poprawność: True

--- Analiza problemu z wielokrotnym dodawaniem ---
Problem: Po wielokrotnym dodawaniu wartość c może przekroczyć zakres,
co może prowadzić do błędów w deszyfrowaniu.
Przykład: c1=644, po trzech dodaniach: c_total=1932
c_total mod p = 18, (c_total mod p) mod 2 = 0


## Szyfrowanie w Pełni Homomorficzne *Fully Homomorphic Encryption*
Szyfrowanie w Pełni Homomorficzne umożliwia realizację wszystkicj operacji arytmetycznych czyli dodawania i mnożenia.

Wykorzystaj bibliotekę Pyfhel w celu sprawdzenia możliwości realizacji operacji matematycznych pod osłoną szyfrowania homomorficznego.

https://pyfhel.readthedocs.io/en/latest/

In [5]:
!pip install pyfhel
import numpy as np
from Pyfhel import Pyfhel, PyPtxt, PyCtxt

HE = Pyfhel()
HE.contextGen(scheme='bfv', n=2**14, t_bits=20)
HE.keyGen()

int1 = np.array([127])
int2 = np.array([-2])
ciphertext1 = HE.encryptInt(int1)
ciphertext2 = HE.encryptInt(int2)


ciphertextSum = ciphertext1 + ciphertext2
summ = HE.decryptInt(ciphertextSum)
print(f"Suma = {summ}")


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Suma = [125   0   0 ...   0   0   0]


In [6]:

HE = Pyfhel()
ckks_params = {
    'scheme': 'CKKS',
    'n': 2**14,
    'scale': 2**30,
    'qi_sizes': [60, 30, 30, 30, 60]
}
HE.contextGen(**ckks_params)
HE.keyGen()
HE.rotateKeyGen()


float1 = np.array([1.2])
float2 = np.array([2.2])
ciphertext1 = HE.encryptFrac(float1)
ciphertext2 = HE.encryptFrac(float2)

ciphertextSum = ciphertext1 + ciphertext2
summ = HE.decryptFrac(ciphertextSum)
print(f"Suma = {summ}")

Suma = [ 3.39999779e+00  8.86778322e-07  1.12801400e-06 ... -3.84346608e-06
  1.71807038e-06 -3.32284520e-06]


### Zadanie 4
Napisz program, który sumuje zawartość tablicy wypełnionej losowymi wartościa całkowitymi z przedziału od 0 do 100. Sumowanie ma być wykonane pod ochroną szyfrowania homomorficznego.

In [7]:
import numpy as np
from Pyfhel import Pyfhel, PyPtxt, PyCtxt
import random

print("=== Zadanie 4: Sumowanie tablicy z użyciem FHE ===")

HE = Pyfhel()
HE.contextGen(scheme='bfv', n=2**14, t_bits=20)
HE.keyGen()

array_size = 10
random_array = [random.randint(0, 100) for _ in range(array_size)]
print(f"\nTablica losowych wartości (0-100):")
print(f"  {random_array}")

plain_sum = sum(random_array)
print(f"\nSuma na jawnych danych: {plain_sum}")

print("\nSzyfrowanie elementów...")
encrypted_array = []
for val in random_array:
    encrypted_val = HE.encryptInt(np.array([val]))
    encrypted_array.append(encrypted_val)

print("Sumowanie zaszyfrowanych wartości...")
encrypted_sum = encrypted_array[0]
for i in range(1, len(encrypted_array)):
    encrypted_sum = encrypted_sum + encrypted_array[i]

decrypted_sum = HE.decryptInt(encrypted_sum)
print(f"\nSuma na zaszyfrowanych danych: {decrypted_sum[0]}")
print(f"Poprawność: {plain_sum == decrypted_sum[0]}")

print("\n--- Alternatywna metoda z wyświetlaniem postępu ---")
encrypted_sum2 = HE.encryptInt(np.array([0]))
for i, val in enumerate(random_array):
    encrypted_val = HE.encryptInt(np.array([val]))
    encrypted_sum2 = encrypted_sum2 + encrypted_val
    partial_sum = HE.decryptInt(encrypted_sum2)
    print(f"  Po dodaniu {val}: suma częściowa = {partial_sum[0]}")

final_sum = HE.decryptInt(encrypted_sum2)
print(f"\nKońcowa suma: {final_sum[0]}")
print(f"Poprawność: {plain_sum == final_sum[0]}")

=== Zadanie 4: Sumowanie tablicy z użyciem FHE ===

Tablica losowych wartości (0-100):
  [47, 71, 78, 78, 97, 67, 68, 68, 74, 86]

Suma na jawnych danych: 734

Szyfrowanie elementów...
Sumowanie zaszyfrowanych wartości...

Suma na zaszyfrowanych danych: 734
Poprawność: True

--- Alternatywna metoda z wyświetlaniem postępu ---
  Po dodaniu 47: suma częściowa = 47
  Po dodaniu 71: suma częściowa = 118
  Po dodaniu 78: suma częściowa = 196
  Po dodaniu 78: suma częściowa = 274
  Po dodaniu 97: suma częściowa = 371
  Po dodaniu 67: suma częściowa = 438
  Po dodaniu 68: suma częściowa = 506
  Po dodaniu 68: suma częściowa = 574
  Po dodaniu 74: suma częściowa = 648
  Po dodaniu 86: suma częściowa = 734

Końcowa suma: 734
Poprawność: True


### Zadanie 5
Oceń wydajność operacji wykonywanych na zaszyfrowanych danych i porównaj z operacjami wykonywamy na wiadomościach jawnych.

In [8]:
import time
import numpy as np
from Pyfhel import Pyfhel
import random

print("=== Zadanie 5: Porównanie wydajności ===")

HE = Pyfhel()
HE.contextGen(scheme='bfv', n=2**14, t_bits=20)
HE.keyGen()

test_size = 100
random_ints = [random.randint(0, 100) for _ in range(test_size)]

print(f"\nTest na {test_size} elementach")

print("\n--- Operacje na jawnych danych ---")
start_time = time.time()
plain_sum = sum(random_ints)
plain_time = time.time() - start_time
print(f"Sumowanie: {plain_time*1000:.4f} ms")
print(f"Wynik: {plain_sum}")

start_time = time.time()
plain_mult = 1
for val in random_ints[:10]:
    plain_mult *= val
plain_mult_time = time.time() - start_time
print(f"Mnożenie (10 elementów): {plain_mult_time*1000:.4f} ms")
print(f"Wynik: {plain_mult}")

print("\n--- Operacje na zaszyfrowanych danych ---")

start_time = time.time()
encrypted_array = []
for val in random_ints:
    encrypted_val = HE.encryptInt(np.array([val]))
    encrypted_array.append(encrypted_val)
encrypt_time = time.time() - start_time
print(f"Szyfrowanie {test_size} elementów: {encrypt_time*1000:.4f} ms")

start_time = time.time()
encrypted_sum = encrypted_array[0]
for i in range(1, len(encrypted_array)):
    encrypted_sum = encrypted_sum + encrypted_array[i]
sum_time = time.time() - start_time
print(f"Sumowanie zaszyfrowanych: {sum_time*1000:.4f} ms")

start_time = time.time()
decrypted_sum = HE.decryptInt(encrypted_sum)
decrypt_time = time.time() - start_time
print(f"Odszyfrowanie: {decrypt_time*1000:.4f} ms")
print(f"Wynik: {decrypted_sum[0]}")

start_time = time.time()
encrypted_mult = encrypted_array[0]
for i in range(1, 10):
    encrypted_mult = encrypted_mult * encrypted_array[i]
mult_time = time.time() - start_time
decrypted_mult = HE.decryptInt(encrypted_mult)
print(f"Mnożenie zaszyfrowanych (10 elementów): {mult_time*1000:.4f} ms")
print(f"Wynik: {decrypted_mult[0]}")

print("\n--- Podsumowanie wydajności ---")
print(f"Stosunek czasu szyfrowania do operacji jawnej: {encrypt_time/plain_time:.2f}x")
print(f"Stosunek czasu sumowania zaszyfrowanego do jawnego: {sum_time/plain_time:.2f}x")
print(f"Stosunek czasu mnożenia zaszyfrowanego do jawnego: {mult_time/plain_mult_time:.2f}x")
print(f"\nCałkowity czas operacji na jawnych danych: {(plain_time + plain_mult_time)*1000:.4f} ms")
print(f"Całkowity czas operacji na zaszyfrowanych danych: {(encrypt_time + sum_time + decrypt_time + mult_time)*1000:.4f} ms")
print(f"Ogólny stosunek: {(encrypt_time + sum_time + decrypt_time + mult_time)/(plain_time + plain_mult_time):.2f}x wolniej")

=== Zadanie 5: Porównanie wydajności ===

Test na 100 elementach

--- Operacje na jawnych danych ---
Sumowanie: 0.0823 ms
Wynik: 5085
Mnożenie (10 elementów): 0.1626 ms
Wynik: 250127680053888000

--- Operacje na zaszyfrowanych danych ---
Szyfrowanie 100 elementów: 3585.9854 ms
Sumowanie zaszyfrowanych: 303.3810 ms
Odszyfrowanie: 29.2053 ms
Wynik: 5085
Mnożenie zaszyfrowanych (10 elementów): 3476.0642 ms
Wynik: 341320

--- Podsumowanie wydajności ---
Stosunek czasu szyfrowania do operacji jawnej: 43596.27x
Stosunek czasu sumowania zaszyfrowanego do jawnego: 3688.32x
Stosunek czasu mnożenia zaszyfrowanego do jawnego: 21377.82x

Całkowity czas operacji na jawnych danych: 0.2449 ms
Całkowity czas operacji na zaszyfrowanych danych: 7394.6359 ms
Ogólny stosunek: 30199.95x wolniej


### Zadanie 6
Napisz program, który oblicza średnią arytmetyczną losowych wartości rzeczywistych z przedziału od 0,0 do 100.0. Obliczenie ma być wykonane pod ochroną szyfrowania homomorficznego.

In [9]:
import numpy as np
from Pyfhel import Pyfhel
import random

print("=== Zadanie 6: Średnia arytmetyczna z użyciem FHE ===")

HE = Pyfhel()
ckks_params = {
    'scheme': 'CKKS',
    'n': 2**14,
    'scale': 2**30,
    'qi_sizes': [60, 30, 30, 30, 60]
}
HE.contextGen(**ckks_params)
HE.keyGen()
HE.rotateKeyGen()

array_size = 10
random_floats = [random.uniform(0.0, 100.0) for _ in range(array_size)]
print(f"\nTablica losowych wartości rzeczywistych (0.0-100.0):")
for i, val in enumerate(random_floats):
    print(f"  [{i}] = {val:.2f}")

plain_mean = sum(random_floats) / len(random_floats)
print(f"\nŚrednia na jawnych danych: {plain_mean:.4f}")

print("\nSzyfrowanie elementów...")
encrypted_array = []
for val in random_floats:
    encrypted_val = HE.encryptFrac(np.array([val]))
    encrypted_array.append(encrypted_val)

print("Sumowanie zaszyfrowanych wartości...")
encrypted_sum = encrypted_array[0]
for i in range(1, len(encrypted_array)):
    encrypted_sum = encrypted_sum + encrypted_array[i]

n = len(random_floats)
one_over_n = 1.0 / n
encrypted_mean = encrypted_sum * one_over_n

decrypted_mean = HE.decryptFrac(encrypted_mean)
print(f"\nŚrednia na zaszyfrowanych danych: {decrypted_mean[0]:.4f}")
print(f"Różnica: {abs(plain_mean - decrypted_mean[0]):.6f}")
print(f"Błąd względny: {abs(plain_mean - decrypted_mean[0]) / plain_mean * 100:.4f}%")

print("\n--- Alternatywna metoda: kodowanie wielu wartości w jednym szyfrogramie ---")
if array_size <= 2**13:
    float_array = np.array(random_floats)
    encrypted_batch = HE.encryptFrac(float_array)
    
    encrypted_sum_batch = encrypted_batch.copy()
    for i in range(1, array_size):
        rotated = HE.rotate(encrypted_batch, i)
        encrypted_sum_batch = encrypted_sum_batch + rotated
    
    encrypted_mean_batch = encrypted_sum_batch * one_over_n
    
    decrypted_mean_batch = HE.decryptFrac(encrypted_mean_batch)
    mean_from_batch = decrypted_mean_batch[0] / array_size
    
    print(f"Średnia z metody batch: {mean_from_batch:.4f}")
    print(f"Różnica: {abs(plain_mean - mean_from_batch):.6f}")

print("\n--- Podsumowanie ---")
print(f"Oryginalna średnia: {plain_mean:.4f}")
print(f"Średnia z FHE (metoda podstawowa): {decrypted_mean[0]:.4f}")
print(f"Poprawność obliczeń homomorficznych została zweryfikowana!")

=== Zadanie 6: Średnia arytmetyczna z użyciem FHE ===

Tablica losowych wartości rzeczywistych (0.0-100.0):
  [0] = 83.47
  [1] = 22.22
  [2] = 59.49
  [3] = 37.12
  [4] = 69.97
  [5] = 93.45
  [6] = 68.26
  [7] = 54.56
  [8] = 91.86
  [9] = 29.67

Średnia na jawnych danych: 61.0067

Szyfrowanie elementów...
Sumowanie zaszyfrowanych wartości...

Średnia na zaszyfrowanych danych: 61.0067
Różnica: 0.000000
Błąd względny: 0.0000%

--- Alternatywna metoda: kodowanie wielu wartości w jednym szyfrogramie ---
Średnia z metody batch: 2.1109
Różnica: 58.895830

--- Podsumowanie ---
Oryginalna średnia: 61.0067
Średnia z FHE (metoda podstawowa): 61.0067
Poprawność obliczeń homomorficznych została zweryfikowana!
